# 07 — Airflow: Walkthrough del Pipeline MLOps

In [ ]:
!pip install apache-airflow pandas -q --quiet 2>/dev/null || echo "Airflow install may take a few minutes..."

## 1. ¿Qué es Apache Airflow?

Apache Airflow es la plataforma de orquestación de workflows más popular en el ecosistema de datos.
Permite definir, programar y monitorear pipelines como código Python.

El pipeline MLOps completo de este curso tiene la siguiente estructura:

```
                    mlops_full_pipeline

generate_data ──▶ validate_data ──▶ preprocess_data ──▶ run_hpo
                                                            │
                                                     train_model
                                                            │
                                                    evaluate_model
                                                            │
                                                    branch_promote
                                                   /              \
                                            promote_model    skip_promote
                                                   \              /
                                               notify_completion
```

Cada nodo es una **Task**. Las flechas son **dependencias**. El conjunto es un **DAG** (Directed Acyclic Graph).

## 2. Conceptos clave de Airflow

| Concepto | Definición |
|----------|------------|
| **DAG** | Directed Acyclic Graph — el pipeline completo, definido en Python |
| **Task** | Unidad de trabajo: un paso del pipeline (ej: `validate_data`) |
| **Operator** | Tipo de task: `PythonOperator`, `BashOperator`, `EmailOperator`, etc. |
| **Scheduler** | Proceso que evalúa cuándo correr DAGs según el `schedule_interval` |
| **Executor** | Proceso que ejecuta las tasks (Local, Celery, Kubernetes) |
| **DAG Run** | Una ejecución concreta del DAG en una fecha específica |
| **Task Instance** | Una ejecución de una Task en un DAG Run específico |
| **XCom** | Mecanismo para pasar datos entre tasks |

**¿Por qué Airflow para MLOps?**
- Define pipelines como código Python versionable en git.
- Reintentos automáticos con `retries` y `retry_delay`.
- UI visual para monitorear ejecuciones y depurar fallos.
- Branching condicional (ej: promover modelo solo si RMSE mejora).
- Integración con MLflow, S3, GCS, Slack, Email, etc.

## 3. Definición de un DAG en código

In [ ]:
# Esto es cómo se define un DAG - no lo ejecutamos aquí
dag_example = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

# 1. Default args - se aplican a todas las tasks
default_args = {
    "owner": "mlops-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": False,
}

# 2. Definir el DAG
with DAG(
    dag_id="mi_pipeline",
    default_args=default_args,
    schedule_interval="0 2 * * 1",  # Lunes a las 2AM (cron)
    start_date=datetime(2024, 1, 1),
    catchup=False,
) as dag:

    # 3. Definir tasks
    task_a = PythonOperator(task_id="task_a", python_callable=mi_funcion_a)
    task_b = PythonOperator(task_id="task_b", python_callable=mi_funcion_b)

    # 4. Definir dependencias
    task_a >> task_b  # task_b corre después de task_a
'''
print(dag_example)

## 4. Expresiones cron para schedule_interval

In [ ]:
cron_examples = {
    "0 2 * * 1":    "Lunes a las 2:00 AM",
    "0 8 * * *":    "Todos los días a las 8:00 AM",
    "*/15 * * * *": "Cada 15 minutos",
    "0 0 1 * *":    "El primer día de cada mes a medianoche",
    "@daily":       "Todos los días a medianoche",
    "@weekly":      "Cada semana (domingo)",
}
print("Guía de expresiones cron:")
print(f"{'Expresión':<20} {'Significado'}")
print("-" * 50)
for expr, meaning in cron_examples.items():
    print(f"{expr:<20} {meaning}")

## 5. La UI de Airflow

Para acceder: http://localhost:8080 (usuario: admin, password: admin)

**VISTA PRINCIPAL (DAGs List):**
```
┌────────────────────────────────────────────────────────────────┐
│  Airflow          DAGs   Browse   Admin   Docs                  │
├────────────────────────────────────────────────────────────────┤
│  ○ Auto Refresh    Search DAGs...                              │
│                                                                 │
│ ┌────────────────┬────────┬───────┬──────┬──────────────────┐ │
│ │ DAG            │Schedule│ Runs  │ Next │ Recent Tasks      │ │
│ ├────────────────┼────────┼───────┼──────┼──────────────────┤ │
│ │ ▶ mlops_full.. │Mon 2AM │   3 ✓ │Mon   │ ●●●●●●●●●○       │ │
│ │ ▶ mlops_monit..│Daily   │  30 ✓ │Tom.  │ ●●●●●●●●●●       │ │
│ │ ▶ mlops_retrai │ None   │   2 ✓ │ —    │ ●●●●●●●●●○       │ │
│ └────────────────┴────────┴───────┴──────┴──────────────────┘ │
│                                                                 │
│ Leyenda: ● success  ● running  ● failed  ○ queued              │
└────────────────────────────────────────────────────────────────┘
```

**VISTA GRAPH (clic en un DAG → pestaña Graph):**
```
┌────────────────────────────────────────────────────────────────┐
│  generate_data                                                  │
│      │  (verde=success, rojo=failed, amarillo=running)          │
│      ▼                                                          │
│  validate_data                                                  │
│      │                                                          │
│      ▼                                                          │
│  preprocess_data ──▶ run_hpo ──▶ train_model ──▶ evaluate...  │
└────────────────────────────────────────────────────────────────┘
```

**Acciones útiles desde la UI:**
- Hacer clic en una task y luego `Clear` para re-ejecutarla sin correr todo el DAG.
- Ver los logs de cada task directamente en el browser.
- Marcar una task como `Success` para saltarla en un run bloqueado.
- Trigger manual de un DAG con parámetros personalizados.

## 6. Simulación del pipeline: ejecutar las funciones manualmente

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import tempfile, os

# Simular el contexto de Airflow
class MockTaskInstance:
    def __init__(self):
        self._xcoms = {}

    def xcom_push(self, key, value):
        self._xcoms[key] = value
        print(f"  XCom push: {key} = {str(value)[:50]}")

    def xcom_pull(self, key, task_ids=None):
        return self._xcoms.get(key)

class MockContext:
    def __init__(self):
        self.ti = MockTaskInstance()
        self.ds = "2024-01-15"

    def __getitem__(self, key):
        return getattr(self, key)

    def get(self, key, default=None):
        return getattr(self, key, default)

print("=== Simulando Task 1: generate_data ===")
rng = np.random.default_rng(42)
timestamps = pd.date_range("2023-01-01", periods=100, freq="h")
df_sim = pd.DataFrame({
    "timestamp": timestamps,
    "demand_kwh": 1000 + 50*np.sin(2*np.pi*timestamps.hour/24) + rng.normal(0, 30, 100),
    "temperature_c": 26 + rng.normal(0, 3, 100),
    "hour": timestamps.hour,
    "day_of_week": timestamps.dayofweek,
    "month": timestamps.month,
})

ctx = MockContext()
ctx.ti.xcom_push("n_records", len(df_sim))
print(f"  Dataset generado: {len(df_sim)} registros")

print("\n=== Simulando Task 2: validate_data ===")
checks = {
    "no_nulls": df_sim.isnull().sum().sum() == 0,
    "demand_positive": (df_sim["demand_kwh"] > 0).all(),
    "hour_range": df_sim["hour"].between(0, 23).all(),
    "temp_range": df_sim["temperature_c"].between(-10, 50).all(),
}
all_passed = all(checks.values())
ctx.ti.xcom_push("validation_passed", all_passed)
for check, result in checks.items():
    print(f"  {'OK' if result else 'FAIL'} {check}")
print(f"  Validacion: {'EXITOSA' if all_passed else 'FALLIDA'}")

## 7. XCom: comunicación entre tasks

XCom (Cross-Communication) es el mecanismo de Airflow para pasar datos entre tasks del mismo DAG Run.

**Cómo funciona:**
1. Task A llama a `ti.xcom_push(key='resultado', value={...})`.
2. Task B (que corre después) llama a `ti.xcom_pull(key='resultado', task_ids='task_a')`.
3. Airflow almacena los valores en su base de datos (SQLite/Postgres).

**Cuándo usarlo:**
- Para pasar metadatos pequeños: run_id, path a un archivo, métricas.
- Para pasar el resultado de una decisión (branch).

**Cuándo NO usarlo:**
- Para datos grandes (>48KB): XCom tiene límites de tamaño.
- Para DataFrames o modelos: en su lugar, guardar en S3/GCS y pasar el path.

In [ ]:
print("XCom (Cross-Communication) permite pasar datos entre tasks")
print()

ctx2 = MockContext()

# Task A pushea datos
def task_a(context):
    result = {"best_rmse": 42.3, "best_params": {"lr": 0.05, "n_est": 300}}
    context["ti"].xcom_push(key="hpo_result", value=result)
    return result

# Task B los recibe
def task_b(context):
    result = context["ti"].xcom_pull(key="hpo_result", task_ids="task_a")
    print(f"  Task B recibió: {result}")
    return result

print("=== Ejecución de Task A ===")
task_a(ctx2)

print("\n=== Ejecución de Task B (recibe XCom de Task A) ===")
task_b(ctx2)

print()
print("NOTA: XCom usa la base de datos de Airflow.")
print("Para datos grandes (>48KB) usar una referencia (S3 path, run_id) en lugar del dato directo.")

## 8. Depurar problemas comunes en Airflow

### Problema 1: DAG no aparece en la UI
**Causas comunes:**
- El archivo `.py` no está en la carpeta `dags/` configurada en `airflow.cfg`.
- Error de sintaxis en el DAG — el Scheduler no puede importar el archivo.
- El DAG tiene `pause_upon_creation=True` o está pausado.

**Solución:**
```bash
python dags/mi_dag.py          # verificar que no hay errores de importación
airflow dags list              # listar DAGs que el scheduler detecta
airflow dags list-import-errors # ver errores de importación
```

### Problema 2: Task falla con ImportError
**Causas comunes:**
- El paquete no está instalado en el entorno del Worker.
- `sys.path` no incluye el directorio correcto.

**Solución:**
```bash
pip install mi_paquete         # en el entorno del worker
# O agregar al requirements.txt del entorno de Airflow
```

### Problema 3: XCom no transfiere el valor
**Causas comunes:**
- La Task A terminó con error antes del `xcom_push`.
- El valor no es serializable (objeto custom, clase, etc.).
- Se usa `xcom_pull` antes de que Task A haya corrido.

**Solución:**
- Serializar el valor a dict/list/str antes de pushear.
- Verificar las dependencias: `task_a >> task_b`.
- En la UI: clic en Task Instance → pestaña XCom para ver los valores.

### Problema 4: Scheduler no dispara el DAG
**Causas comunes:**
- `start_date` está en el futuro.
- `catchup=True` y hay muchas fechas históricas pendientes.
- El Scheduler process no está corriendo.

**Solución:**
```bash
airflow scheduler              # arrancar el scheduler
airflow dags trigger mi_dag    # disparar manualmente para probar
# Revisar: start_date debe ser en el pasado, catchup=False para DAGs nuevos
```

## 9. Disparar un DAG via REST API

In [ ]:
import json

# Así se dispara un DAG via REST API (sin ejecutar realmente)
print("Para disparar un DAG via REST API:")
print()
print("import requests")
print()
print("response = requests.post(")
print('    "http://localhost:8080/api/v1/dags/mlops_full_pipeline/dagRuns",')
print('    auth=("admin", "admin"),')
print('    headers={"Content-Type": "application/json"},')
print('    json={"conf": {"triggered_by": "notebook", "note": "manual trigger"}},')
print(")")
print("print(response.json())")
print()
print("# O para ver el estado:")
print("# GET http://localhost:8080/api/v1/dags/mlops_full_pipeline/dagRuns")

## Resumen

En este notebook aprendimos:

1. **Airflow**: plataforma de orquestación que define pipelines como código Python (DAGs).
2. **Conceptos clave**: DAG, Task, Operator, Scheduler, Executor, XCom.
3. **Definición de DAGs**: `with DAG(...) as dag:` con `default_args`, `schedule_interval`, `start_date`.
4. **Cron**: expresiones para programar ejecuciones (`0 2 * * 1` = lunes 2AM).
5. **UI**: vistas de DAGs List y Graph para monitorear y depurar ejecuciones.
6. **XCom**: mecanismo para pasar metadatos entre tasks (usar paths para datos grandes).
7. **Debugging**: los 4 problemas más comunes y cómo resolverlos.
8. **REST API**: cómo disparar DAGs programáticamente desde cualquier sistema externo.

**Próximo notebook**: `08_monitoring_drift.ipynb` — detectar data drift y concept drift con Evidently AI.